In [ ]:
!apt-get -qq install -y wget gdal-bin

import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import subprocess

In [ ]:
!rm -rf /content/drive
from google.colab import drive
drive.mount('/content/drive')

In [ ]:

CHIRTS_ERA5_BASES = {
    "tmin": "ftp://ftp.chc.ucsb.edu/pub/org/chc/experimental/CHIRTS-ERA5/tmin/tifs/monthly",
     "tmax": "ftp://ftp.chc.ucsb.edu/pub/org/chc/experimental/CHIRTS-ERA5/tmax/tifs/monthly",
}

CHIRTS_ERA5_PREFIXES = {
    "tmin": "Tmin",
    "tmax": "Tmax",
}

VARIABLES = ["tmin", "tmax"]
START_DATE = "2018-01-01"
END_DATE   = "2023-12-31"

# California bounding box
CA_BBOX = (-124.48, 32.53, -114.13, 42.01)
# format: (xmin, ymin, xmax, ymax)

def build_chirts_era5_url(date: str, variable: str) -> str:
    dt = pd.Timestamp(date)
    fname = f"CHIRTS-ERA5.monthly_{CHIRTS_ERA5_PREFIXES[variable]}.{dt:%Y.%m}.tif"
    return f"{CHIRTS_ERA5_BASES[variable]}/{fname}"

def crop_to_bbox(infile, outfile, bbox):
    xmin, ymin, xmax, ymax = bbox

    result = subprocess.run(
        [
            "gdalwarp",
            "-te", str(xmin), str(ymin), str(xmax), str(ymax),
            "-overwrite",
            str(infile),
            str(outfile),
        ],
        capture_output=True,
        text=True
    )

    return result.returncode == 0, result.stderr.strip()

def download_chirts_monthly_tifs(
    start_date,
    end_date,
    variables,
    out_dir="chirts_era5_monthly_tifs",
    crop_bbox=None
):
    dates = pd.date_range(start=start_date, end=end_date, freq="MS")
    out_dir = Path(out_dir)
    rows = []

    for var in variables:
        raw_dir = out_dir / var / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)

        if crop_bbox is not None:
            cropped_dir = out_dir / var / "cropped_ca"
            cropped_dir.mkdir(parents=True, exist_ok=True)
        else:
            cropped_dir = None

        for dt in tqdm(dates, desc=f"Downloading {var}"):
            url = build_chirts_era5_url(dt.strftime("%Y-%m-%d"), var)
            file_name = url.split("/")[-1]

            raw_file_path = raw_dir / file_name

            result = subprocess.run(
                ["wget", "-nv", "-O", str(raw_file_path), url],
                capture_output=True,
                text=True
            )

            downloaded_ok = result.returncode == 0
            cropped_ok = None
            cropped_file_path = None

            if not downloaded_ok:
                print("FAILED DOWNLOAD:", url)
                print(result.stderr.strip())

            if downloaded_ok and crop_bbox is not None:
                cropped_file_path = cropped_dir / file_name
                cropped_ok, crop_msg = crop_to_bbox(raw_file_path, cropped_file_path, crop_bbox)

                if not cropped_ok:
                    print("FAILED CROP:", raw_file_path)
                    print(crop_msg)

            rows.append({
                "date": dt.normalize(),
                "year": dt.year,
                "month": dt.month,
                "variable": var,
                "raw_file_path": str(raw_file_path),
                "cropped_file_path": str(cropped_file_path) if cropped_file_path else None,
                "url": url,
                "downloaded": downloaded_ok,
                "cropped": cropped_ok,
            })

    return pd.DataFrame(rows)

download_manifest = download_chirts_monthly_tifs(
    START_DATE,
    END_DATE,
    VARIABLES,
    out_dir="/content/drive/MyDrive/Untitled Folder/chirts_era5_monthly_tifs",
    crop_bbox=CA_BBOX
)

download_manifest.to_csv("chirts_era5_monthly_download_manifest_2018_2023.csv", index=False)

In [ ]:
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
import subprocess

CA_BBOX = (-124.48, 32.53, -114.13, 42.01)

CHIRTS_ERA5_DAILY_BASES = {
    "hi":   "https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/hi/tifs/daily",
    "wbgt": "https://data.chc.ucsb.edu/experimental/CHIRTS-ERA5/wbgt/tifs/daily"
}

CHIRTS_ERA5_DAILY_PREFIXES = {
    "hi": "HI",
    "wbgt": "WBGT"
}

DAILY_VARIABLES = [
                  # "hi"
                  "wbgt"
                   ]
DAILY_START_DATE = "2023-5-29"
DAILY_END_DATE   = "2023-12-31"

def crop_to_bbox(infile, outfile, bbox):
    xmin, ymin, xmax, ymax = bbox

    result = subprocess.run(
        [
            "gdalwarp",
            "-te", str(xmin), str(ymin), str(xmax), str(ymax),
            "-overwrite",
            str(infile),
            str(outfile),
        ],
        capture_output=True,
        text=True
    )

    return result.returncode == 0, result.stderr.strip()

def build_chirts_era5_daily_url(date: str, variable: str) -> str:
    dt = pd.Timestamp(date)
    prefix = CHIRTS_ERA5_DAILY_PREFIXES[variable]

    fname = f"{prefix}.{dt:%Y.%m.%d}.tif"
    return f"{CHIRTS_ERA5_DAILY_BASES[variable]}/{dt:%Y}/{fname}"

def download_chirts_daily_tifs(
    start_date,
    end_date,
    variables,
    out_dir="chirts_era5_daily_tifs",
    crop_bbox=None
):
    dates = pd.date_range(start=start_date, end=end_date, freq="D")
    out_dir = Path(out_dir)
    rows = []

    for var in variables:
        raw_dir = out_dir / var / "raw"
        raw_dir.mkdir(parents=True, exist_ok=True)

        if crop_bbox is not None:
            cropped_dir = out_dir / var / "cropped_ca"
            cropped_dir.mkdir(parents=True, exist_ok=True)
        else:
            cropped_dir = None

        for dt in tqdm(dates, desc=f"Downloading daily {var}"):
            url = build_chirts_era5_daily_url(dt.strftime("%Y-%m-%d"), var)
            file_name = url.split("/")[-1]
            raw_file_path = raw_dir / file_name

            result = subprocess.run(
                ["wget", "-nv", "-O", str(raw_file_path), url],
                capture_output=True,
                text=True
            )

            downloaded_ok = result.returncode == 0
            cropped_ok = None
            cropped_file_path = None

            if not downloaded_ok:
                print("FAILED DOWNLOAD:", url)
                print(result.stderr.strip())

            if downloaded_ok and crop_bbox is not None:
                cropped_file_path = cropped_dir / file_name
                cropped_ok, crop_msg = crop_to_bbox(raw_file_path, cropped_file_path, crop_bbox)

                if not cropped_ok:
                    print("FAILED CROP:", raw_file_path)
                    print(crop_msg)

            rows.append({
                "date": dt.normalize(),
                "year": dt.year,
                "month": dt.month,
                "day": dt.day,
                "variable": var,
                "raw_file_path": str(raw_file_path),
                "cropped_file_path": str(cropped_file_path) if cropped_file_path else None,
                "url": url,
                "downloaded": downloaded_ok,
                "cropped": cropped_ok,
            })

    return pd.DataFrame(rows)

download_manifest = download_chirts_daily_tifs(
    DAILY_START_DATE,
    DAILY_END_DATE,
    DAILY_VARIABLES,
    out_dir="/content/drive/MyDrive/Untitled Folder/chirts_era5_daily_tifs",
    crop_bbox=CA_BBOX
)